# Data Preparation

The following code block first reads the VCQ data from its CSV file into the `raw_df` DataFrame object. It then creates the demo_df object to contain only the columns for demographic questions. In addition, it removes the prefix 'demo_' from all column names in demo_df.

Review the [VCQ code book](https://docs.google.com/document/d/1uYfUuUsPVQyD3puVvbIMjamYuVPr_f4gtJuSB5Wj-WA/edit) as needed.

In [2]:
# Collect data into a DataFrame object
import pandas as pd
raw_df = pd.read_csv('https://raw.githubusercontent.com/qarnac/cs201/main/vcq_data_nan.csv')

# Create a new DataFrame object to holds only demographic data
demo_df = raw_df.filter(regex='^demo').copy()
# Remove the 'demo_' prefix from column names
demo_df.columns = demo_df.columns.str.removeprefix('demo_')
# Replace 1 - 4 with respective class standings
class_map = {1:'First-year', 2:'Sophomore', 3:'Junior', 4:'Senior'}
demo_df['classstanding'] = demo_df['classstanding'].replace(class_map)
# Replace 1 with 'Yes' and 2 with 'No' for related columns
yes_no_map = {1:'Yes', 2:'No'}
yes_no_columns = ['children', 'employed01', 'employed02', 'employed04']
demo_df[yes_no_columns] = demo_df[yes_no_columns].replace(yes_no_map)
demo_df.head()

,gender,age,relationshipstatus01,children,classstanding,units,gpa,employed01,employed02,employed03,incomehousehold,incomepersonal,politics,religion,relationshipstatus02,employed04
0,1,19.0,1,No,Sophomore,15.0,3.15,Yes,Yes,45.0,60000.0,16000.0,NaN,6.0,1.0,Yes
1,2,26.0,2,No,Junior,14.0,3.30,Yes,Yes,35.0,27000.0,27000.0,4.0,5.0,2.0,Yes
2,1,NaN,1,No,Junior,15.0,3.11,No,No,NaN,6000.0,2000.0,4.0,3.0,1.0,NaN
3,2,NaN,1,No,Sophomore,17.0,3.40,No,No,NaN,NaN,NaN,3.0,4.0,1.0,No
4,2,19.0,1,No,Sophomore,13.0,3.00,No,No,NaN,NaN,NaN,3.0,3.0,1.0,No


## Practice

In [ ]:
# Replace 1 with 'Male', 2 with 'Female', and 3 with 'Other' for the gender column
# Use the value_counts function to find out the number of students for each category


In [ ]:
# Replace 1 with 'Single' and 2 with 'In relationship' for the relationshipstatus01 and relationshipstatus02 column
# Use the value_counts function to find out the number of students for each category


# Drop columns

We can see from the above exploration that some columns are duplicated. That is, they always contain the same data. A careful review of the [VCQ code book](https://docs.google.com/document/d/1uYfUuUsPVQyD3puVvbIMjamYuVPr_f4gtJuSB5Wj-WA/edit) revealed that Dr. Trinidad ask two questions twice in the survey:
*  demo_relationshipstatus01 (in the demographics section) and demo_relationshipstatus02 (in the Conflict Resolution section) are the same question.
*  demo_employed02 (in the demographics section) and demo_employed04 (in the job security section) are the same question.

We can easily drop columns by using the `drop` function as shown in the following code block. Inside the function call,
*  The columns parameter specifies a list of columns to be dropped.
*  The inplace parameter needs to be set to True if

In [3]:
demo_df = demo_df.drop(columns=['relationshipstatus02', 'employed04'])
demo_df.columns

Index(['gender', 'age', 'relationshipstatus01', 'children', 'classstanding',
       'units', 'gpa', 'employed01', 'employed02', 'employed03',
       'incomehousehold', 'incomepersonal', 'politics', 'religion'],
      dtype='object')

## Practice

In [ ]:
# Drop the relationshipstatus01 and children columns from demo_df
# Use the columns attribute to show the remaining columns


# Missing *data*

Under the `'age'` column we can see some rows with **'NaN'** value. **NaN** means Not a Number. This is an indication that the student did not answer that question. The `pandas` library supports a `count` function that can counts the number of non-missing values in each column.

In [ ]:
# count the number of non-NaN values in one column
demo_df['gpa'].count()

np.int64(518)

In [ ]:
# count the number of non-NaN values in multiple columns
demo_df[['politics', 'religion']].count()

,0
politics,505
religion,539


The above results show that only 518 students answered the GPA question, 505 for the politics question and 539 for the religion question. We know that there are 543 rows in the dataset. Therefore, the above results indicate that 25 students didn't answer the GPA question, 38 students didn't answer the politics question, and only 4 students didn't answer the religion question.

Missing values (NaN) are common in real world datasets, and how we handle them depends on the type of analysis we want to perform. In many cases, we simply keep NaN values because many pandas functions ignore them by default when computing statistics. When missing data must be addressed, we have a few options. One option is to drop rows with NaNs, which is straightforward using the dropna() function. However, such action risks losing valuable information if many records contain missing entries. Therefore, we want to be careful in making such decision.

For example, if we our focus of the analyses is the determining whethere there's any difference of GPA among students from religion groups, it's important to create a new DataFrame object to focus on the two columns before dropping the rows of NaN values.  

In [ ]:
# Create a new DataFrame object to include only the religion and gpa columns
# Note the copy function is used so that changes made to gpa_df will not affect demo_df
gpa_df = demo_df[['religion', 'gpa']].copy()
gpa_df = gpa_df.dropna()
gpa_df.count()

,0
religion,515
gpa,515


From the above result, we can see that only 515 rows are left in the dataset after NaNs are dropped. Looking back, we saw 518 students answered the gpa question and 329 students answered the religion question. Why do we now only have 515 rows left? This is because only 515 students answered both gpa and religion questions.

## Practice

In [ ]:
# Does a student's job impact their studies?
# What columns from demo_df are relevant for this question?
# Create a new DataFrame object with the relevant columns only and call the dropna function to drop
# all rows with NaN values for these columns. Use the count function to find out the number of
# remaining non-NaN values for each column.


In [ ]:
# Think of another question you would like to know about the students based on the demographics data
# Create a new DataFrame object with the relevant columns only and call the dropna function to drop
# all rows with NaN values for these columns. Use the shape attribute to find out the number of rows and columns
# of the updated dataset.


# Rename columns

Some of the column names are not as descriptive as they could be. For example, how are these three columns different, `employed01`, `employed02`, `employed03`?

We had used the renamed function in the past. The following code block gives a quick review:
*  It sets up a map to show what new name to be used for a column.
*  The rename function is called with the columns parameter set to the map.
*  The result of the rename function is assigned back to the DataFrame object.

In [ ]:
name_map = {'employed01':'employed_past_12'}
demo_df = demo_df.rename(columns=name_map)
demo_df.columns

## Practice

In [ ]:
# Rename the other two employed columns with more descriptive names
# Use the columns attribute to show the updated column list


# Row filtering

This is not a new skill. We can comparison operators and logical operators to set up filtering conditions so that we don't have to work with all 543 students everytime. Here are some examples for review

In [ ]:
condition = (demo_df['age'] < 21) & (demo_df['units'] >= 12)
under_21_full_time_df = demo_df[condition].copy()
under_21_full_time_df.head()

## Practice

In [ ]:
# Create a DataFrame object to hold responses from students who responded being currently employed


In [ ]:
# Create a DataFrame object to hold responses from students whose GPAs are under 3.0


In [ ]:
# Create a DataFrame object to hold responses from students who took between 12 and 16 units in that semester


In [ ]:
# Create a DataFrame object to hold responses from students who are Female Sophomores


# The cut function

This is also a skill we learned before. We can set up related labels and bin boundaries to classify values of a numeric column into categorical values. Here is a quick example for review.

In [ ]:
units_labels = ['Part Time', 'Full Time', 'Overload']
units_bins = [0, 9, 17, 30]
demo_df['units_category'] = pd.cut(demo_df['units'],
                                 labels = units_labels,
                                 bins = units_bins,
                                 right=True)
demo_df[['units', 'units_category']].tail()

## Practice

In [ ]:
# Create a new column to classify the values in the politics column into conservative, moderate, and liberal.


In [ ]:
# Create a new column to classify the values in gpa columns into bad, good, amazing
